+ 심리학/인지언어학에서 점화효과.
+ 딥러닝 계층을 깊게 쌓는 것과 비슷하다.
+ 깊게 쌓을 수록 더 추상적인 생각. 그렇다면 얼마나 깊게 쌓아야하는가?
+ 너무 깊게 쌓으면 사람이 딴 생각을 하듯, 딥러닝 네트워크 역시 다른 생각으로 빠질 수도 있을 것 같다.

<br>

> 문맥의 문제!! 단어의 모호성/중의성 등을 이미지에 기반한 priming effect로 해결할 수 있지 않을까???

In [1]:
import os
os.sys.path.append('../../official_github/')

In [2]:
import numpy as np
import matplotlib.pyplot as plt

from common.multi_layer_net_extend import MultiLayerNetExtend
from common.optimizer import SGD, Adam

In [3]:
test_dict = {}
train_dict = {}
root_path = './cifar-10-batches-py/'


def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

accepted_keys = [b'labels', b'data'] ## select keys from [b'batch_label', b'labels', b'data', b'filenames']
for dir in os.listdir(root_path):
    if dir == 'batches.meta':
        pass
    elif dir == 'test_batch':
        test_batch = unpickle(os.path.join(root_path, dir))
        test_dict.update((k,np.array(v)) for k,v in test_batch.items() if k in accepted_keys)
    else:
        train_batch = unpickle(os.path.join(root_path, dir))
        if not train_dict:
            train_dict.update((k,np.array(v)) for k,v in train_batch.items() if k in accepted_keys)
        else:
            for key in accepted_keys:
                train_dict[key] = np.concatenate((train_dict[key], train_batch[key]), axis=0)

In [4]:
x_train, t_train = train_dict[b'data'], train_dict[b'labels']
x_test, t_test = test_dict[b'data'], test_dict[b'labels']

In [5]:
# max_epochs = 20
# train_size = x_train.shape[0]
# batch_size = 100
# iter_per_epoch = max(train_size / batch_size, 1)

# learning_rate = 0.01

# hidden_size_list = [100]


# net = MultiLayerNetExtend(input_size=3072,
#                           hidden_size_list=[100],
#                           output_size=10,
#                           use_batchnorm=True)
# optimizer = Adam(lr=learning_rate)

# train_acc_list = []
# test_acc_list = []

# iters = 100000

# for i in range(iters):
#     batch_mask = np.random.choice(train_size, batch_size)
#     x_batch = x_train[batch_mask]
#     t_batch = t_train[batch_mask]
    
#     grads = net.gradient(x_batch, t_batch)
#     optimizer.update(net.params, grads)
    
#     if i % iter_per_epoch == 0:
#         train_acc = net.accuracy(x_train, t_train)
#         test_acc = net.accuracy(x_test, t_test)
#         train_acc_list.append(train_acc)
#         test_acc_list.append(test_acc)
#         print(f'train_acc: {train_acc}, test_acc: {test_acc}')


In [6]:
max_epochs = 20
train_size = x_train.shape[0]
batch_size = 100

iters = 50_000
iter_per_epoch = max(train_size / batch_size, 1)

learning_rate = 0.01

network_dict = {'Affine_2': MultiLayerNetExtend(input_size=3072, hidden_size_list=[1000], output_size=10, weight_init_std='relu', weight_decay_lambda=0.1, use_batchnorm=True, use_dropout=True),
                'Affine_3': MultiLayerNetExtend(input_size=3072, hidden_size_list=[1000, 100], output_size=10, weight_init_std='relu', weight_decay_lambda=0.1, use_batchnorm=True, use_dropout=True),
                'Affine_4': MultiLayerNetExtend(input_size=3072, hidden_size_list=[1000, 100, 50], output_size=10, weight_init_std='relu', weight_decay_lambda=0.1, use_batchnorm=True, use_dropout=True),
                }

output_dict = {}

for key, network in network_dict.items():
    net = network
    optimizer = Adam(lr=learning_rate)

    output_dict[f'{key} train acc'] = []
    output_dict[f'{key} test acc'] = []
    
    print(f'=== {key} network training start ===')
    for i in range(iters):
        batch_mask = np.random.choice(train_size, batch_size)
        x_batch = x_train[batch_mask]
        t_batch = t_train[batch_mask]

        grads = net.gradient(x_batch, t_batch)
        optimizer.update(net.params, grads)

        if i % iter_per_epoch == 0:
            train_acc = net.accuracy(x_train, t_train)
            test_acc = net.accuracy(x_test, t_test)
            output_dict[f'{key} train acc'].append(train_acc)
            output_dict[f'{key} test acc'].append(test_acc)
            print(f'=== {key} result report ===')
            print(f'train_acc: {train_acc}, test_acc: {test_acc}')
    
    
    

=== Affine_2 network training start ===
=== Affine_2 result report ===
train_acc: 0.13398, test_acc: 0.1321
=== Affine_2 result report ===
train_acc: 0.14722, test_acc: 0.1459
=== Affine_2 result report ===
train_acc: 0.15474, test_acc: 0.1577


KeyboardInterrupt: 